### 실습 환경 준비

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from enum import Enum
from typing import Any

load_dotenv("../.env")

True

In [2]:
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"prompt": "passage: ", "normalize_embeddings": True},
    query_encode_kwargs={"prompt": "query: ", "normalize_embeddings": True},
)

db = Chroma(
    persist_directory="./chroma_db_e5",
    collection_name="langchain-docs",
    embedding_function=embeddings,
    create_collection_if_not_exists=False,
)

c:\Users\limux\Documents\1. Projects\llm-engineer-roadmap\02-rag-agents-langchain-langgraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1605.07it/s]


In [3]:
model = ChatOpenAI(model="gpt-6-luna")
retriever = db.as_retriever()

class QueryGenerationOutput(BaseModel):
    queries: list[str] = Field(..., description="검색 쿼리 목록")


prompt = ChatPromptTemplate.from_template('''
다음 문맥만을 고려해 질문에 답하세요.

문맥: """
{context}
"""

질문: {question}
''')

query_generation_prompt = ChatPromptTemplate.from_template("""\
질문에 대해 벡터 데이터베이스에서 관련 문서를 검색하기 위한
3개의 서로 다른 검색 쿼리를 생성하세요.
거리 기반 유사성 검색의 한게를 극복하기 위해
사용자의 질문에 대해 여러 관점을 제공하는 것이 목표입니다.

질문: {question}
""")

query_generation_chain = (
    query_generation_prompt
    | model.with_structured_output(QueryGenerationOutput)
    | (lambda x: x.queries)
)

### LLM에 의한 라우팅

In [4]:
# 먼저 LangChain 공식 문서를 검색하는 Retriever(langchain_document_retriever)와 웹 검색 Retriever(web_retriever)를 준비

from langchain_community.retrievers import TavilySearchAPIRetriever

langchain_document_retriever = retriever.with_config({"run_name": "langchain_document_retriever"})
web_retriever = TavilySearchAPIRetriever(k=3).with_config({"run_name": "web_retriever"})

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_39012\3875684307.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import TavilySearchAPIRetriever


In [5]:
# 다음으로 사용자의 입력을 기반으로 LLM이 Retriever를 선택하는 Chain(route_chain)을 구현
# LangChain의 with_structured_output을 사용해 Retriever를 선택하는 Chain은 다음과 같다.

class Route(str, Enum):
    langchain_document = "langchain_document"
    web = "web"

class RouteOutput(BaseModel):
    route: Route

route_prompt = ChatPromptTemplate.from_template("""
질문에 답변하기 위해 적절한 Retirever를 선택하세요.

질문: {question}
""")

route_chain = (
    route_prompt
    | model.with_structured_output(RouteOutput)
    | (lambda x: x.route)
)

In [8]:
# 마지막으로 라우팅 결과를 고려해 검색하는 routed_retriever 함수와 전체 처리 흐름의 Chain(route_rag_chain)을 구현
def routed_retriever(inp: dict[str, Any]) -> list[Document]:
    question = inp["question"]
    route = inp["route"]

    if route == Route.langchain_document:
        return langchain_document_retriever.invoke(question)
    elif route == Route.web:
        return web_retriever.invoke(question)

    raise ValueError(f"Unknown route: {route}")

route_rag_chain = (
    {
        "question": RunnablePassthrough(),
        "route": route_chain
    }
    | RunnablePassthrough.assign(context=routed_retriever)
    | prompt | model | StrOutputParser()
)

route_rag_chain.invoke("LangChain의 개요를 알려줘")

'LangChain은 대규모 언어 모델(LLM)을 활용한 애플리케이션을 개발하기 위한 프레임워크입니다. 모델, 임베딩, 벡터 저장소 등 여러 구성 요소와 통합을 연결하고, 표준 인터페이스를 제공해 AI 애플리케이션 개발을 쉽게 돕습니다.\n\n주요 특징은 다양한 데이터 소스를 연결해 정보를 보강하고, 필요에 따라 모델을 바꿔 사용할 수 있으며, 여러 통합 기능을 활용할 수 있다는 점입니다. 생태계에는 평가와 관측을 위한 LangSmith와 복잡한 에이전트를 구성하는 LangGraph가 포함됩니다. 에이전트 개발을 처음 시작하거나 더 높은 수준의 추상화를 원한다면 LangChain의 사전 구축된 에이전트 기능을 사용할 수 있습니다.'

In [9]:
# 이번에는 “오늘 서울 날씨는?”이라고 질문해 보자
route_rag_chain.invoke("오늘 서울 날씨는?")

'오늘 서울은 오후 5시 기준 13°C로 화창하고 비 올 확률은 0%입니다. 저녁에는 맑은 날씨가 이어지며 기온이 오후 11시쯤 7°C까지 내려갈 전망입니다. 북서풍이 불어 쌀쌀하니 겉옷을 챙기세요.'

### 하이브리드 검색 구현

In [4]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    path="./langchain-docs",
    glob="**/*.mdx",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    use_multithreading=True,
)

documents = loader.load()

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_30052\2037659830.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [5]:
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-base")
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"원본 문서: {len(documents)}")
print(f"분할된 청크: {len(chunks)}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4739 > 512). Running this sequence through the model will result in indexing errors


원본 문서: 2532
분할된 청크: 18413


In [6]:
# LangChain 공식 문서에 대해 임베딩 벡터의 유사도 검색과 BM25를 사용한 검색을 조합하기로 한다.
# 임베딩 벡터의 유사도 검색 Retriever(chroma_retriever)와 BM25를 사용한 검색 Retriever(bm25_retriever)를 준비
from langchain_community.retrievers import BM25Retriever

chroma_retriever = retriever.with_config({"run_name": "chroma_retriever"})
bm25_retriever = BM25Retriever.from_documents(chunks).with_config({"run_name": "bm25_retriever"})

In [7]:
def reciprocal_rank_fusion(ranked_lists: list[list[Document]], k: int=60) -> list[str]:
    scores = {}
    for documents in ranked_lists:
        for rank, document in enumerate(documents, start=1):
            content = document.page_content
            scores[content] = scores.get(content, 0.0) + 1 / (k + rank)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [content for content, _ in ranked]

In [8]:
# 그리고 이 두 가지 검색을 모두 실행하는 hybrid_retriever를 구현
hybrid_retriever = (
    RunnableParallel(
        {"chroma_documents": chroma_retriever, "bm25_documents": bm25_retriever}
    )
    | (lambda x: [x["chroma_documents"], x["bm25_documents"]])
    | reciprocal_rank_fusion
)

In [10]:
# hybrid_retriever를 사용한 RAG의 Chain을 구현하고 실행
hybrid_rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": hybrid_retriever,
    }
    | prompt | model | StrOutputParser()
)

hybrid_rag_chain.invoke("LangChain의 개요를 알려줘")

'**LangChain**은 대규모 언어 모델(LLM)을 활용한 애플리케이션을 개발하기 위한 프레임워크입니다. 모델, 임베딩, 벡터 저장소 등 여러 구성 요소에 표준화된 인터페이스를 제공하고, 이들을 연결해 애플리케이션을 만들 수 있도록 돕습니다.\n\n주요 특징은 다음과 같습니다.\n\n- 다양한 데이터 소스와 연결해 모델에 실시간 정보를 제공할 수 있습니다.\n- 구성 요소를 바꾸기 쉬워 여러 모델과 통합을 유연하게 활용할 수 있습니다.\n- 일반적인 LLM 및 도구 호출 흐름을 위한 에이전트 기능을 제공합니다.\n\n생태계에는 애플리케이션의 평가와 관찰성을 지원하는 **LangSmith**, 복잡하고 상태를 유지하는 에이전트의 실행 흐름을 세밀하게 조정하는 **LangGraph**가 있습니다. LangGraph는 LangChain 구성 요소와 함께 사용할 수 있지만, LangChain 없이도 사용할 수 있습니다.'